In [1]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 25.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatibl

In [1]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
from ortools.sat.python import cp_model

In [3]:
# ==========================================
# BLOQUE: DEFINICIÓN DE PERFILES (DICCIONARIO MAESTRO)
# ==========================================

data_personal = [
    # JEFATURA Y COORDINACIÓN (Seguimiento Mañana 4 semanas)
    {
        "Nombre": "Lic. Garcia", "Rol": "Jefe", "Puede_Noche": False, "Max_Noches": 0,
        "Turnos_Prohibidos_LV": ["Tarde"], "Semanas_Seguimiento": {"Mañana": 4},
        "Asignaciones_Fijas": []
    },
    {
        "Nombre": "Lic. Toledo", "Rol": "Coordinador", "Puede_Noche": False, "Max_Noches": 0,
        "Turnos_Prohibidos_LV": ["Tarde"], "Semanas_Seguimiento": {"Mañana": 4},
        "Asignaciones_Fijas": []
    },
    {
        "Nombre": "Lic. Franco", "Rol": "Coordinador", "Puede_Noche": False, "Max_Noches": 0,
        "Turnos_Prohibidos_LV": ["Tarde"], "Semanas_Seguimiento": {"Mañana": 4},
        "Asignaciones_Fijas": []
    },
    {
        "Nombre": "Lic. Moyano", "Rol": "Coordinador", "Puede_Noche": False, "Max_Noches": 0,
        "Turnos_Prohibidos_LV": ["Tarde"], "Semanas_Seguimiento": {"Mañana": 4},
        "Asignaciones_Fijas": []
    },

    # KINESIOLOGÍA NOCTURNA
    {
        "Nombre": "Lic. Juarez", "Rol": "Nocturno", "Puede_Noche": True, "Max_Noches": 12,
        "Turnos_Prohibidos_LV": ["Mañana", "Tarde", "Mañana_especial", "Tarde_especial"],
        "Semanas_Seguimiento": {}, "Asignaciones_Fijas": []
    },

    # CASOS ESPECIALES
    {
        "Nombre": "Lic. Giaccoppo", "Rol": "DRotativo", "Puede_Noche": True, "Max_Noches": 4,
        "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Mañana": 1, "Tarde": 2},
        "Asignaciones_Fijas": [
            {"Dia": "Lunes",     "Turno": "Tarde_especial", "Tipo": "Especial",    "Horas": 6},
        ]
    },
    {
        "Nombre": "Lic. Camargo N.", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4,
        "Turnos_Prohibidos_LV": ["Mañana", "Mañana_especial"], "Semanas_Seguimiento": {"Indistinto": 1},
        "Asignaciones_Fijas": [
            {"Dia": "Lunes",     "Turno": "Tarde_especial", "Tipo": "Especial",    "Horas": 6},
            {"Dia": "Miercoles", "Turno": "Tarde",          "Tipo": "Asistencial", "Horas": 6},
            {"Dia": "Viernes",   "Turno": "Tarde",          "Tipo": "Asistencial", "Horas": 6},
            {"Dia": "Domingo",   "Turno": "Dia",            "Tipo": "Asistencial", "Horas": 12}
        ],
        "Notas": "Lunes tarde: Base de datos (Especial). Domingo: Guardia 12hs. No mañanas L-V."
    },
    {
        "Nombre": "Lic. Coniglio",
        "Rol": "Rotativo",
        "Puede_Noche": True,
        "Max_Noches": 4,
        "Turnos_Prohibidos_LV": [],
        "Semanas_Seguimiento": {"Indistinto": 1},
        "Asignaciones_Fijas": [
            {"Dia": "Miercoles", "Turno": "Mañana_especial", "Tipo": "Especial", "Horas": 6},
            {"Dia": "Miercoles", "Turno": "Tarde_especial",  "Tipo": "Especial", "Horas": 6}
        ],
        "Notas": "Todos los miércoles: Mañana y Tarde_especial (12hs fijas)."
    },

    # ROTATIVOS COMUNES (1 Semana Seguimiento Indistinto)
    { "Nombre": "Lic. Guzman", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Leonforte", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Marino", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Mesa", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Sosa", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Syriani", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Vander", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Vivas", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Espinosa", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Welch", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Guardia", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
    { "Nombre": "Lic. Flores", "Rol": "Rotativo", "Puede_Noche": True, "Max_Noches": 4, "Turnos_Prohibidos_LV": [], "Semanas_Seguimiento": {"Indistinto": 1}, "Asignaciones_Fijas": [] },
]

df_personal = pd.DataFrame(data_personal)

In [4]:
# Crear el DataFrame
df_personal = pd.DataFrame(data_personal)

# Pequeña validación: Ver cuántas horas fijas tiene ya asignadas Camargo
def calcular_horas_fijas(asignaciones):
    return sum(item['Horas'] for item in asignaciones)

df_personal['Horas_Fijas_Semanales'] = df_personal['Asignaciones_Fijas'].apply(calcular_horas_fijas)

In [5]:
# Definimos la cantidad de profesionales necesarios y las horas de cada turno
# Esto le dice al algoritmo cuántos "huecos" vacíos debe llenar cada día

demanda_turnos = {
    "Semana": { # Lunes a Viernes
        "Mañana":         {"Horas": 6,  "Vacantes": 6},
        "Tarde":          {"Horas": 6,  "Vacantes": 4},
        "Noche":          {"Horas": 12, "Vacantes": 2},
        "Mañana_especial": {"Horas": 6,  "Vacantes": 0}, # <--- IA no rellena
        "Tarde_especial":  {"Horas": 6,  "Vacantes": 0}  # <--- IA no rellena
    },
    "Finde_Feriado": { # Sábados, Domingos y Feriados
        "Dia":   {"Horas": 12, "Vacantes": 3},
        "Noche":  {"Horas": 12, "Vacantes": 2}
    }
}


In [6]:
# 1. Inicializamos el modelo (El lienzo en blanco)
modelo = cp_model.CpModel()

# 2. Definimos los parámetros de tiempo (Bloque de 4 semanas)
DIAS_DEL_BLOQUE = 28

# Los nombres de los turnos según si es día de semana o finde
turnos_semana = ["Mañana", "Tarde", "Noche", "Mañana_especial", "Tarde_especial"]
turnos_finde = ["Dia", "Noche"]

# --- NUEVO: Diccionario traductor de días de Texto a Número ---
mapa_dias = {
    "Lunes": 0, "Martes": 1, "Miercoles": 2, "Jueves": 3,
    "Viernes": 4, "Sabado": 5, "Domingo": 6
}

# 3. Creamos el diccionario que guardará todos los "interruptores"
turnos = {}

# Recorremos cada persona en tu tabla df_personal
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    asignaciones = persona.get('Asignaciones_Fijas', []) # Rescatamos sus turnos fijos

    # Recorremos cada uno de los 28 días
    for dia in range(DIAS_DEL_BLOQUE):
        # Asumimos que el Día 0 es un Lunes.
        dia_semana = dia % 7
        es_finde = (dia_semana == 5) or (dia_semana == 6)
        lista_turnos = turnos_finde if es_finde else turnos_semana

        # Creamos un interruptor por cada turno de ese día para esa persona
        for t in lista_turnos:
            turnos[(nombre, dia, t)] = modelo.NewBoolVar(f'turno_{nombre}_dia{dia}_{t}')

            # --- NUEVO: Inyectar Asignaciones Fijas ---
            # Si este turno 't' en este día de la semana coincide con una asignación fija, lo clavamos en 1 (Trabaja sí o sí)
            for asig in asignaciones:
                # El replace(" ", "_") es por si en el perfil lo dejaste con espacio, lo iguala.
                turno_limpio = asig['Turno'].replace(" ", "_")

                if mapa_dias.get(asig['Dia']) == dia_semana and turno_limpio == t:
                    modelo.Add(turnos[(nombre, dia, t)] == 1)

In [7]:
feriados = [] # Lista vacía por ahora para que no tire error

In [8]:
# ==========================================
# BLOQUE 1: REGLAS INQUEBRANTABLES
# ==========================================

# 1. REGLA DINÁMICA: LÍMITES DE NOCHES Y TURNOS PROHIBIDOS (Reemplaza reglas Juarez, Jefes y Camargo)
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']

    # A) Noches
    noches_vble = [turnos[(nombre, d, "Noche")] for d in range(28) if (nombre, d, "Noche") in turnos]
    if not persona.get('Puede_Noche', True):
        for n in noches_vble:
            modelo.Add(n == 0)
    elif noches_vble:
        modelo.Add(sum(noches_vble) <= int(persona.get('Max_Noches', 4)))

    # B) Turnos Prohibidos L-V (Lee directo del diccionario)
    prohibidos_lv = persona.get('Turnos_Prohibidos_LV', [])
    for d in range(28):
        if (d % 7) < 5: # Solo Lunes a Viernes
            for t_prohibido in prohibidos_lv:
                if (nombre, d, t_prohibido) in turnos:
                    modelo.Add(turnos[(nombre, d, t_prohibido)] == 0)

# 2. REGLA DINÁMICA: SEMANAS DE SEGUIMIENTO (4 turnos iguales en la semana)
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    seguimientos = persona.get('Semanas_Seguimiento', {})

    if seguimientos: # Si el diccionario no está vacío para esta persona
        for tipo_seg, cant_semanas in seguimientos.items():
            semanas_activas = [] # Guardará booleanos (1 o 0) de cada semana

            for sem in range(4):
                dias_lv = range(sem * 7, sem * 7 + 5)
                m_norm = [turnos[(nombre, d, "Mañana")] for d in dias_lv if (nombre, d, "Mañana") in turnos]
                t_norm = [turnos[(nombre, d, "Tarde")] for d in dias_lv if (nombre, d, "Tarde") in turnos]

                # Creamos un interruptor maestro que dice "¿Cumple seguimiento esta semana?"
                cumple_sem = modelo.NewBoolVar(f'seg_{tipo_seg}_{nombre}_sem{sem}')

                if tipo_seg == "Mañana":
                    modelo.Add(sum(m_norm) >= 4).OnlyEnforceIf(cumple_sem)
                elif tipo_seg == "Tarde":
                    modelo.Add(sum(t_norm) >= 4).OnlyEnforceIf(cumple_sem)
                elif tipo_seg == "Indistinto":
                    # Si es indistinto, puede cumplirlo por la mañana O por la tarde
                    cumple_m = modelo.NewBoolVar(f'seg_ind_m_{nombre}_sem{sem}')
                    cumple_t = modelo.NewBoolVar(f'seg_ind_t_{nombre}_sem{sem}')
                    modelo.Add(sum(m_norm) >= 4).OnlyEnforceIf(cumple_m)
                    modelo.Add(sum(t_norm) >= 4).OnlyEnforceIf(cumple_t)
                    modelo.AddBoolOr([cumple_m, cumple_t]).OnlyEnforceIf(cumple_sem)

                semanas_activas.append(cumple_sem)

            # Obligamos a que la cantidad de semanas que cumplió sea exacta a lo pedido en su perfil
            modelo.Add(sum(semanas_activas) == cant_semanas)

# 3. COBERTURA DINÁMICA (No se tocó, está perfecto)
for dia in range(28):
    es_f = (dia % 7) >= 5 or dia in feriados
    tipo_dia = "Finde_Feriado" if es_f else "Semana"
    for t_nombre, info in demanda_turnos[tipo_dia].items():
        pool = [turnos[(p['Nombre'], dia, t_nombre)] for _, p in df_personal.iterrows() if (p['Nombre'], dia, t_nombre) in turnos]
        if pool and info["Vacantes"] > 0:
            modelo.Add(sum(pool) == info["Vacantes"])

# 4. LÍMITE DE 36 HORAS SEMANALES (No se tocó, está perfecto)
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    for sem in range(4):
        horas_semanales = []
        for d in range(sem * 7, (sem + 1) * 7):
            es_finde = (d % 7) >= 5 or d in feriados
            for t in (turnos_finde if es_finde else turnos_semana):
                if (nombre, d, t) in turnos:
                    h = 12 if (es_finde or t == "Noche") else 6
                    horas_semanales.append(turnos[(nombre, d, t)] * h)
        if horas_semanales:
            modelo.Add(sum(horas_semanales) <= 36)

# 5. DESCANSO POST-NOCHE (No se tocó, está perfecto)
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    for d in range(27):
        if (nombre, d, "Noche") in turnos:
            siguiente_dia = [turnos[(nombre, d+1, t)] for t in (turnos_semana + turnos_finde) if (nombre, d+1, t) in turnos]
            for t_sig in siguiente_dia:
                modelo.AddImplication(turnos[(nombre, d, "Noche")], t_sig.Not())

# 6. REGLA INCOMPATIBILIDAD DÍA/NOCHE (No se tocó, está perfecto)
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    for d in range(28):
        if (nombre, d, "Noche") in turnos:
            turnos_dia = [turnos[(nombre, d, t)] for t in ["Mañana", "Tarde", "Mañana especial", "Tarde especial", "Dia"] if (nombre, d, t) in turnos]
            for t_dia in turnos_dia:
                modelo.Add(turnos[(nombre, d, "Noche")] + t_dia <= 1)

# 7. REGLA EXCLUSIÓN MUTUA (Mañana vs Especial) (No se tocó, está perfecto)
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    for d in range(28):
        if (nombre, d, "Mañana") in turnos and (nombre, d, "Mañana especial") in turnos:
            modelo.Add(turnos[(nombre, d, "Mañana")] + turnos[(nombre, d, "Mañana especial")] <= 1)
        if (nombre, d, "Tarde") in turnos and (nombre, d, "Tarde especial") in turnos:
            modelo.Add(turnos[(nombre, d, "Tarde")] + turnos[(nombre, d, "Tarde especial")] <= 1)

In [9]:
# ==========================================
# 6. OPTIMIZACIÓN Y EQUIDAD (MENSUAL + ANUAL)
# ==========================================

# 1. PRIMERO: Definimos todas las variables de control
max_findes_empleado = modelo.NewIntVar(0, 8, 'max_findes')

# Variables de equidad del MES
max_horas_mes = modelo.NewIntVar(0, 200, 'max_horas_mes')
min_horas_mes = modelo.NewIntVar(0, 200, 'min_horas_mes')

# Variables de equidad del AÑO
max_anual = modelo.NewIntVar(0, 5000, 'max_anual')
min_anual = modelo.NewIntVar(0, 5000, 'min_anual')

# 2. SEGUNDO: Aplicamos el "piso" de horas para que nadie trabaje de menos
modelo.Add(min_horas_mes >= 100)

puntos_seguimiento = []
puntos_combo_finde = []

# 3. BUCLE DE PERSONAS
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    rol = persona['Rol']
    turnos_finde_persona = []
    horas_mes = []

    for semana in range(4):
        # A. PREMIOS POR SEGUIMIENTO FLEXIBLE (Fomentar constancia L-V)
        lunes_a_viernes = range(semana * 7, semana * 7 + 5)
        turnos_a_evaluar = ["Mañana"] if rol == "Coordinador" else turnos_semana

        for t in turnos_a_evaluar:
            dias_trabajados = [turnos[(nombre, d, t)] for d in lunes_a_viernes if (nombre, d, t) in turnos]
            if dias_trabajados:
                puntos_seguimiento.extend(dias_trabajados)

        # B. PREMIOS POR COMBO FINDE (Sábado y Domingo en el mismo turno)
        dia_sabado = semana * 7 + 5
        dia_domingo = semana * 7 + 6
        for t_finde in turnos_finde:
            if (nombre, dia_sabado, t_finde) in turnos and (nombre, dia_domingo, t_finde) in turnos:
                combo = modelo.NewBoolVar(f'combo_{nombre}_sem{semana}_{t_finde}')
                # AddMinEquality es la forma estricta de decir: combo = 1 si y solo si Sabado=1 Y Domingo=1
                modelo.AddMinEquality(combo, [turnos[(nombre, dia_sabado, t_finde)], turnos[(nombre, dia_domingo, t_finde)]])
                puntos_combo_finde.append(combo)

        # C. RECOLECTAR HORAS DEL MES
        for d in range(semana * 7, (semana + 1) * 7):
            es_finde = (d % 7) >= 5 or d in feriados
            list_t = turnos_finde if es_finde else turnos_semana
            for t in list_t:
                if (nombre, d, t) in turnos:
                    h = 12 if (es_finde or t == "Noche") else 6
                    horas_mes.append(turnos[(nombre, d, t)] * h)

    # Límite de fines de semana para que sean parejos
    for d in range(28):
        es_f = (d % 7) >= 5 or d in feriados
        if es_f:
            for t_f in turnos_finde:
                if (nombre, d, t_f) in turnos:
                    turnos_finde_persona.append(turnos[(nombre, d, t_f)])

    if turnos_finde_persona:
        modelo.Add(sum(turnos_finde_persona) <= max_findes_empleado)

    # D. DOBLE CANDADO DE EQUIDAD (Mensual y Anual) - APLICA A TODOS
    total_mes = sum(horas_mes)
    total_anual_proyectado = persona.get('Horas_Anuales_Previas', 0) + total_mes

    # Eliminamos el filtro de roles para que todos entren en la repartición
    # El total de este mes debe estar entre el min y max del grupo general
    modelo.Add(total_mes <= max_horas_mes)
    modelo.Add(total_mes >= min_horas_mes)

    # El acumulado anual también debe estar entre el min y max proyectado general
    modelo.Add(total_anual_proyectado <= max_anual)
    modelo.Add(total_anual_proyectado >= min_anual)

# 4. CÁLCULO DE BRECHAS
brecha_mensual = modelo.NewIntVar(0, 200, 'brecha_mensual')
modelo.Add(brecha_mensual == max_horas_mes - min_horas_mes)
modelo.Add(brecha_mensual <= 24) # Regla de oro: máximo 24hs de diferencia

brecha_anual = modelo.NewIntVar(0, 1000, 'brecha_anual')
modelo.Add(brecha_anual == max_anual - min_anual)

# 5. LA FUNCIÓN OBJETIVO (El "Cerebro" de la IA)
modelo.Minimize(
    (max_findes_empleado * 500) +    # Prioridad 1: Minimizar el máximo de findes por persona
    (brecha_anual * 100) +           # Prioridad 2: Emparejar el histórico
    (brecha_mensual * 50) -          # Prioridad 3: Emparejar el mes
    (sum(puntos_seguimiento) * 5) -  # Prioridad 4: Fomentar semanas de seguimiento
    (sum(puntos_combo_finde) * 15)   # Prioridad 5: Fomentar Sáb+Dom juntos
)


In [10]:
# ==========================================
# 7. CANDADO ABSOLUTO A TURNOS ESPECIALES
# ==========================================
lista_blanca_especiales = []
dias_map_inv = {0: "Lunes", 1: "Martes", 2: "Miercoles", 3: "Jueves", 4: "Viernes", 5: "Sabado", 6: "Domingo"}

# 1. Anotamos quién, qué día y en qué turno tiene permiso explícito
for index, persona in df_personal.iterrows():
    nombre = persona['Nombre']
    for asig in persona.get('Asignaciones_Fijas', []):
        if "especial" in asig['Turno']:
            for d in range(28):
                if dias_map_inv[d % 7] == asig['Dia'] and d not in feriados:
                    lista_blanca_especiales.append((nombre, d, asig['Turno']))

# 2. Aplastamos a cero a todo el que intente entrar y no esté en la lista
for (nombre, d, t), variable in turnos.items():
    if t in ["Mañana_especial", "Tarde_especial"]:
        if (nombre, d, t) not in lista_blanca_especiales:
            modelo.Add(variable == 0)

In [11]:
# ==========================================
# ¡RESOLVER EL ROMPECABEZAS!
# ==========================================
solver = cp_model.CpSolver()
# Le damos 60 segundos a la IA para que encuentre la justicia para Juarez
solver.parameters.max_time_in_seconds = 90
status = solver.Solve(modelo)

print("⏳ Resolviendo el cronograma con todas las reglas y preferencias...")
status = solver.Solve(modelo)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("✅ ¡CRONOGRAMA GENERADO!")

    # 1. DEFINIMOS LA FECHA REAL DE INICIO (El Día 0, que debe ser Lunes)
    fecha_inicio = pd.to_datetime("2024-04-01")

    resultados = []
    dias_nombres = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

    for dia in range(DIAS_DEL_BLOQUE):
        # 2. CALCULAMOS LA FECHA EXACTA DE ESTE DÍA
        fecha_actual = fecha_inicio + pd.Timedelta(days=dia)
        dia_semana = dias_nombres[fecha_actual.weekday()]

        # 3. AGREGAMOS LOS FERIADOS ACÁ TAMBIÉN
        es_finde = (dia % 7) >= 5 or dia in feriados
        tipos_turnos = turnos_finde if es_finde else turnos_semana

        for t in tipos_turnos:
            for index, persona in df_personal.iterrows():
                nombre = persona['Nombre']
                if solver.Value(turnos[(nombre, dia, t)]) == 1:
                    resultados.append({
                        "Fecha": fecha_actual.strftime("%Y-%m-%d"), # <--- FECHA REAL
                        "Dia_Semana": dia_semana,
                        "Turno": t,
                        "Kinesiologo": nombre
                    })

    df_resultados = pd.DataFrame(resultados)
    display(df_resultados.head(15)) # Mostramos los primeros 15 turnos
else:
    print("❌ INVIABLE: El motor no pudo encontrar una solución. Las reglas son muy estrictas.")

⏳ Resolviendo el cronograma con todas las reglas y preferencias...
❌ INVIABLE: El motor no pudo encontrar una solución. Las reglas son muy estrictas.


In [ ]:
# ==========================================
# EXPORTACIÓN MAESTRA A EXCEL (Orden + Color)
# ==========================================

# 1. Definimos la jerarquía visual (quién aparece arriba en la celda)
jerarquia = {
    "Lic. Garcia": 1,
    "Lic. Franco": 2,
    "Lic. Moyano": 3,
    "Lic. Toledo": 4
}

def ordenar_kinesiologos(lista_nombres):
    # Los jefes van arriba (1, 2, 3...), el resto al final (100)
    lista_ordenada = sorted(lista_nombres, key=lambda x: jerarquia.get(x, 100))
    return "\n".join(lista_ordenada)

# 2. Creamos la tabla pivot usando la FECHA real como columnas
df_pivot = df_resultados.groupby(['Turno', 'Fecha'])['Kinesiologo'].apply(ordenar_kinesiologos).unstack()

# 3. Ordenamos las filas (Turnos) de forma lógica
orden_turnos = ["Mañana", "Mañana_especial", "Tarde", "Tarde_especial", "Dia", "Noche"]
turnos_existentes = [t for t in orden_turnos if t in df_pivot.index]
df_pivot = df_pivot.reindex(turnos_existentes)

# 4. Exportamos con formato profesional
file_name = 'Cronograma_Servicio_Kinesiologia.xlsx'
with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
    df_pivot.to_excel(writer, sheet_name='Cronograma')

    workbook  = writer.book
    worksheet = writer.sheets['Cronograma']

    # Definimos los formatos
    header_fmt = workbook.add_format({'bold': True, 'bg_color': '#D7E4BD', 'border': 1, 'align': 'center'})
    cell_fmt = workbook.add_format({'text_wrap': True, 'valign': 'top', 'border': 1, 'font_size': 9})

    # Colores por turno para el formato condicional
    colores = {
        "Mañana": '#EBF1DE',
        "Tarde":  '#FEF2CB',
        "Dia":    '#DAEEF3',
        "Noche":  '#E5E0EC'
    }

    # Ajustamos columnas (Turnos + 28 días)
    worksheet.set_column(0, 0, 15, header_fmt)
    worksheet.set_column(1, 28, 18, cell_fmt)

    # 5. Aplicamos colores dinámicos según el nombre del turno en la primera columna
    for i, turno in enumerate(turnos_existentes):
        fila_excel = i + 1
        color = colores.get(turno.split('_')[0], '#FFFFFF') # El split es por si es "Mañana_especial"
        worksheet.set_row(fila_excel, 40, workbook.add_format({'bg_color': color, 'text_wrap': True, 'border': 1, 'valign': 'top'}))

print(f"✅ ¡Excel generado con éxito! Archivo: {file_name}")

✅ ¡Excel generado con éxito! Archivo: Cronograma_Servicio_Kinesiologia.xlsx


In [ ]:
# ==========================================
# REPORTE DE CONTROL: HORAS Y FINDES LIBRES
# ==========================================

# 1. Definimos cuántas horas vale cada turno
def asignar_horas(turno):
    # Si el turno incluye la palabra "Noche" o es el turno de "Dia" de finde/feriado, son 12hs
    if turno in ["Noche", "Dia"]:
        return 12
    else: # Mañanas y Tardes (comunes o especiales) valen 6hs
        return 6

# Creamos una copia para no alterar los resultados originales
df_reporte = df_resultados.copy()
df_reporte['Horas'] = df_reporte['Turno'].apply(asignar_horas)

# Agrupamos para sumar las horas totales de cada profesional
horas_por_persona = df_reporte.groupby('Kinesiologo')['Horas'].sum().reset_index()

# 2. Calculamos los fines de semana libres
# Aseguramos que la columna Fecha sea formato datetime de Pandas
df_reporte['Fecha'] = pd.to_datetime(df_reporte['Fecha'])

# Filtramos SOLO los turnos que caen en Sábado (5) o Domingo (6)
df_findes = df_reporte[df_reporte['Fecha'].dt.dayofweek.isin([5, 6])]

# Contamos en cuántas semanas distintas trabajó un fin de semana
findes_trabajados = df_findes.groupby('Kinesiologo')['Fecha'].apply(
    lambda x: x.dt.isocalendar().week.nunique()
).reset_index()
findes_trabajados.columns = ['Kinesiologo', 'Findes_Trabajados']

# 3. Unimos las tablas (Horas + Findes)
reporte_final = pd.merge(horas_por_persona, findes_trabajados, on='Kinesiologo', how='left')

# Si alguien tiene "NaN" en Findes_Trabajados, significa que trabajó 0 findes
reporte_final['Findes_Trabajados'] = reporte_final['Findes_Trabajados'].fillna(0)

# Asumimos que el bloque tiene 4 fines de semana en total
TOTAL_FINDES = 4
reporte_final['Findes_Libres'] = TOTAL_FINDES - reporte_final['Findes_Trabajados']

# 4. Limpiamos y ordenamos para mostrar
reporte_final = reporte_final[['Kinesiologo', 'Horas', 'Findes_Libres']]
reporte_final = reporte_final.sort_values(by='Horas', ascending=False).reset_index(drop=True)

print("📊 REPORTE DE CONTROL: HORAS Y FINES DE SEMANA LIBRES")
display(reporte_final)

# Opcional: También lo exportamos a CSV por si quieres guardarlo
reporte_final.to_csv('Reporte_Horas_Kinesiologia.csv', index=False)

📊 REPORTE DE CONTROL: HORAS Y FINES DE SEMANA LIBRES


,Kinesiologo,Horas,Findes_Libres
0,Lic. Garcia,132,2
1,Lic. Toledo,126,2
2,Lic. Franco,120,2
3,Lic. Moyano,120,2
4,Lic. Syriani,108,3
5,Lic. Flores,108,3
6,Lic. Espinosa,108,2
7,Lic. Coniglio,108,2
8,Lic. Guzman,108,2
9,Lic. Leonforte,108,3
